# 01: Keyword Generation
=====================

Generate search queries for YouTube data collection to find hard case demand signals.

**Core Strategy:** Cover all product categories that may need protection cases / storage solutions.

**Query Structure:** Product + protection/storage/carry scenario

**Output:** `../data/shared_data/query_assignments/`

In [1]:
import itertools
import pandas as pd
import re
from pathlib import Path
from datetime import datetime

## Configuration

In [2]:
KEYWORD_CONFIG = {

    "object_categories": {

        "photography": ["camera", "DSLR", "mirrorless camera", "camera lens", "GoPro", "drone"],

        "musical_instruments": ["guitar", "violin", "drums", "keyboard piano", "synthesizer", "DJ controller"],

        "outdoor_camping": ["tent", "sleeping bag", "camping cookware", "fishing gear", "climbing gear", "portable stove"],

        "travel_luggage": ["luggage", "suitcase", "travel bag", "toiletry bag", "cable organizer", "packing cube"],

        "gaming_console": ["Steam Deck", "Nintendo Switch", "PlayStation", "gaming headset", "VR headset", "gaming mouse"],

        "tools_equipment": ["power tools", "drill", "oscilloscope", "soldering iron", "3D printer", "multimeter"],

        "collectibles": ["figurine", "action figure", "PVC statue", "anime figure", "Lego set", "vinyl record"],

        "electronic_accessories": ["hard drive", "SSD", "external storage", "USB drive", "laptop", "power bank"],

        "eyewear_optics": ["sunglasses", "eyeglasses", "goggles", "binoculars", "telescope", "spectacles"],

        "kitchen_cookware": ["chef knife", "kitchen knife set", "cast iron skillet", "wine glass", "espresso machine", "blender"],

        "sports_fitness": ["golf clubs", "tennis racket", "bicycle", "skateboard", "surfboard", "dumbbells"],

        "medical_health": ["CPAP machine", "hearing aid", "blood pressure monitor", "glucose meter", "inhaler", "first aid kit"],

        "baby_kids": ["stroller", "car seat", "baby monitor", "toys", "building blocks", "kids camera"],

        "instrument_accessories": ["pedal", "amplifier", "effects pedal", "instrument cable", "tuner", "metronome"],

        "drone_rc": ["drone", "RC car", "RC plane", "FPV drone", "racing drone", "RC helicopter"],

        "beauty_personal": ["hair dryer", "straightener", "curling iron", "electric toothbrush", "shaver", "skincare device"],

        "art_crafts": ["paint set", "art supplies", "sketchbook", "crafting tools", "sewing machine", "embroidery kit"],

        "jewelry_watches": ["watch", "jewelry", "necklace", "bracelet", "ring", "earrings"],
    },

    # ============================================================
    # Query Patterns（精简至 6 个，覆盖保护壳需求信号）
    # ============================================================
    "protection_queries": [
        "{obj} hard case",
        "{obj} protective case"
    ],

    "storage_queries": [
        "{obj} storage case",
        "{obj} carry case"
    ],

    "usage_queries": [
        "{obj} everyday carry",
        "{obj} EDC setup"
    ],

    "problem_queries": [
        "{obj} damaged",
        "{obj} best case for"
    ],

    "leader_discovery_queries": [
        "hard case review",
        "protective case for valuable items",
        "best carry case",
        "storage solution for fragile items",
        "gear organizer",
        "equipment storage",
        "cables organizer",
        "tech organizer bag",
        "travel fragile items",
        "how to protect fragile gear",
        "packing fragile equipment",
        "fragile items travel case",
        "best protective case",
        "shockproof case review",
        "waterproof case for gear",
        "crush proof case",
        "impact protection case",
        "damage protection case",
        "safe transport case",
        "hard shell case review",
        "protective case comparison",
        "best hard case brand",
        "durable case for expensive equipment",
        "heavy duty protective case",
        "compact storage case",
        "organizer case review",
        "carrying case recommendation",
        "travel case for valuables",
        "portable case solution",
        "fragile equipment storage",
        "professional gear case",
        "storage case for hobby",
        "travel setup for gear",
        "EDC carry case",
        "everyday carry organizer",
        "gear bag setup",
        "kit bag organization",
        "packing solution for equipment",
        "how to pack fragile gear",
        "travel packing tips gear",
        "best packing method fragile",
        "moving fragile items",
        "shipping fragile equipment",
        "best case for camera gear",
        "best case for camera lens",
        "best case for drone",
        "best case for guitar",
        "best case for violin",
        "best case for keyboard",
        "best case for synthesizer",
        "best case for DJ controller",
        "best case for laptop",
        "best case for hard drive",
        "best case for Steam Deck",
        "best case for Nintendo Switch",
        "best case for gaming headset",
        "best case for power tools",
        "best case for drill",
        "best case for oscilloscope",
        "best case for soldering iron",
        "best case for tent",
        "best case for sleeping bag",
        "best case for fishing gear",
        "best case for headphones",
        "best case for figurines",
        "best case for collectibles",
        "best case for vinyl records",
        "best case for art supplies",
        "best case for makeup",
        "best case for tools",
        "best case for fragile items",
        "best protective case for travel",
        "best storage case organization",
        "how to store camera equipment",
        "how to store guitar at home",
        "how to transport drone",
        "how to pack fragile decorations",
        "how to ship collectibles safely",
        "moving fragile belongings",
        "best way to pack fragile items",
        "travel fragile belongings protection"
    ],

    "negative_terms": [
        "iphone case", "phone case", "smartphone case",
        "android phone", "samsung case", "google pixel",
        "ipad case", "tablet case",
        "fashion case", "cute case", "aesthetic case",
        "decorative case", "phone skin",
        "clothing", "shoes", "handbag",
        "software", "app", "download"
    ]
}

## Generation Functions

In [3]:
def filter_queries(df, negative_terms):
    df = df.copy()
    df["query_lower"] = df["query"].str.lower()
    
    def contains_negative(text):
        for term in negative_terms:
            if ' ' in term:
                if term in text:
                    return True
            else:
                pattern = r'\b' + re.escape(term) + r'\b'
                if re.search(pattern, text):
                    return True
        return False
    
    mask = ~df["query_lower"].apply(contains_negative)
    return df[mask].drop(columns=["query_lower"]).drop_duplicates()

In [4]:
def generate_all_queries(config):
    rows = []
    
    query_types = {
        "protection": config["protection_queries"],
        "storage": config["storage_queries"],
        "usage": config["usage_queries"],
        "problem": config["problem_queries"]
    }
    
    for cat_name, objects in config["object_categories"].items():
        for obj in objects:
            for query_type, patterns in query_types.items():
                for pattern in patterns:
                    rows.append({
                        "category": cat_name,
                        "object": obj,
                        "query": pattern.format(obj=obj),
                        "type": query_type
                    })
    
    df = pd.DataFrame(rows)
    return filter_queries(df, config["negative_terms"])

In [5]:
def generate_leader_queries(config):
    rows = []
    for q in config["leader_discovery_queries"]:
        rows.append({"query": q, "type": "discovery"})
    df = pd.DataFrame(rows)
    return filter_queries(df, config["negative_terms"])

## Member Assignment

In [6]:
MEMBER_ASSIGNMENT = {
    "yufei": ["photography", "sports_fitness", "eyewear_optics"],
    "kesong": ["gaming_console", "electronic_accessories", "kitchen_cookware"],
    "xinchen": ["musical_instruments", "instrument_accessories", "medical_health"],
    "shuyue": ["tools_equipment", "drone_rc", "art_crafts", "beauty_personal"],
    "zeman": ["outdoor_camping", "travel_luggage", "collectibles", "jewelry_watches"]
}

## Generate & Save

In [7]:
output_dir = Path("../data/shared_data/query_assignments")
output_dir.mkdir(parents=True, exist_ok=True)

# No date in filename

all_tables = []

all_queries_df = generate_all_queries(KEYWORD_CONFIG)
print(f"Generated {len(all_queries_df):,} member queries")

leader_df = generate_leader_queries(KEYWORD_CONFIG)
print(f"Generated {len(leader_df):,} leader queries")

for member, cats in MEMBER_ASSIGNMENT.items():
    member_df = all_queries_df[all_queries_df["category"].isin(cats)].copy()
    member_df["assigned_to"] = member
    
    output_path = output_dir / f"{member}_query.csv"
    member_df.to_csv(output_path, index=False)
    print(f"  {member}: {len(member_df):,} queries")
    all_tables.append(member_df)

leader_df = leader_df.copy()
leader_df["assigned_to"] = "leader"
leader_output = output_dir / f"qianyin.csv"
leader_df.to_csv(leader_output, index=False)
print(f"  qianyin: {len(leader_df):,} queries")
all_tables.append(leader_df)

Generated 864 member queries
Generated 81 leader queries
  yufei: 144 queries
  kesong: 144 queries
  xinchen: 144 queries
  shuyue: 192 queries
  zeman: 192 queries
  qianyin: 81 queries


## Statistics

In [8]:
all_combined = pd.concat(all_tables)

print("\n" + "="*60)
print("QUERY STATISTICS")
print("="*60)
print(f"\nTotal: {len(all_combined):,} queries")
print(f"\nBy Query Type:")
print(all_combined["type"].value_counts())
print(f"\nBy Category:")
print(all_combined["category"].value_counts())


QUERY STATISTICS

Total: 897 queries

By Query Type:
type
protection    204
storage       204
usage         204
problem       204
discovery      81
Name: count, dtype: int64

By Category:
category
photography               48
tools_equipment           48
collectibles              48
travel_luggage            48
outdoor_camping           48
art_crafts                48
beauty_personal           48
drone_rc                  48
instrument_accessories    48
eyewear_optics            48
medical_health            48
musical_instruments       48
kitchen_cookware          48
electronic_accessories    48
gaming_console            48
sports_fitness            48
jewelry_watches           48
Name: count, dtype: int64


## Save Simplified Version

In [9]:
simple_dir = output_dir / "simple"
simple_dir.mkdir(exist_ok=True)

for member in MEMBER_ASSIGNMENT.keys():
    df = pd.read_csv(output_dir / f"{member}_query.csv")
    df[["query"]].drop_duplicates().to_csv(
        simple_dir / f"{member}_query_only.csv",
        index=False
    )

leader_df[["query"]].to_csv(
    simple_dir / "qianyin_query_only.csv",
    index=False
)

print(f"Simplified CSV saved to: {simple_dir}")

Simplified CSV saved to: ..\data\shared_data\query_assignments\simple
